In [1]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
import logging

# Configure logging to overwrite "_.log" every run (mode="w")
logging.basicConfig(
    level=logging.INFO,
    format="%(message)s",
    handlers=[
        logging.FileHandler("q2_a.log", mode="w"),  # "w" overwrites, "a" appends
        logging.StreamHandler()
    ],
    force=True  # Resets any existing logging handlers
)
logger = logging.getLogger(__name__)

seed=260007

def fit_samme_adaboost(X_flat, y, n_estimators, max_depth):

    N = len(y)
    classes = np.unique(y)
    K = len(classes)

    w = np.ones(N, dtype=np.float64) / N

    estimators = []
    estimator_weights = []

    history = {
        "eps_ts": [],
        "alphas": [],
        "max_sample_weights": [],
        "top10_weights": [],
        "sample_weights_over_time": []
    }

    logger.info("=" * 65)
    logger.info(f"Initial sample weights (w_1):\n{w}\n")
    logger.info("=" * 65 + "\n")
    

    for t in range(n_estimators):
        logger.info(f"LOGGING SAMME ROUND {t}")
        clf = DecisionTreeClassifier(max_depth=max_depth, random_state=seed + t)
        clf.fit(X_flat, y, sample_weight=w)

        preds = clf.predict(X_flat)
        incorrect = (preds != y)

        eps_t = np.sum(w * incorrect) / np.sum(w)

        random_guess_error = 1.0 - (1.0 / K)
        if eps_t >= random_guess_error:
            if t == 0:
                alpha_t = 1e-10
                estimators.append(clf)
                estimator_weights.append(alpha_t)
            break

        if eps_t == 0:
            alpha_t = 100.0
        else:
            alpha_t = np.log((1.0 - eps_t) / eps_t) + np.log(K - 1)

        w_updated = w * np.exp(alpha_t * incorrect)

        w_next = w_updated / np.sum(w_updated)

        logger.info(f"Weak Learner Predictions h_1(x):\n{preds}")
        logger.info(f"Ground Truth y:\n{y}")
        logger.info(f"Misclassified Indicator (1[h_1(x) != y]):\n{incorrect.astype(int)}")
        logger.info(f"\nWeighted Error (eps_1): {eps_t:.4f}")
        logger.info(f"Learner Weight (alpha_1):  {alpha_t:.4f}")
        logger.info(f"\nUpdated & Normalized Sample Weights (w_2):\n{w_next}")
        logger.info("=" * 65 + "\n")

        estimators.append(clf)
        estimator_weights.append(alpha_t)

        history["eps_ts"].append(eps_t)
        history["alphas"].append(alpha_t)
        history["max_sample_weights"].append(np.max(w_next))
        
        top_k = 10
        history["top10_weights"].append(np.sum(np.sort(w_next)[-top_k:]))
        history["sample_weights_over_time"].append(w_next.copy())

        w = w_next

    return estimators, estimator_weights, classes, history

rng = np.random.default_rng(seed=seed)

X_toy = rng.integers(0, 12, size=(10, 2))

y_toy = rng.integers(0, 2, size=10)

logger.info(f"Toy Dataset Size: {X_toy.shape[0]} samples, {len(np.unique(y_toy))} classes (K=3)\n")

estimators, estimator_weights, classes, history = fit_samme_adaboost(
    X_flat=X_toy,
    y=y_toy,
    n_estimators=1,
    max_depth=4,
)

logging.shutdown()

Toy Dataset Size: 10 samples, 2 classes (K=3)

Initial sample weights (w_1):
[0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]


LOGGING SAMME ROUND 0
Weak Learner Predictions h_1(x):
[1 0 0 0 1 1 0 1 0 1]
Ground Truth y:
[1 0 0 0 1 1 1 1 0 1]
Misclassified Indicator (1[h_1(x) != y]):
[0 0 0 0 0 0 1 0 0 0]

Weighted Error (eps_1): 0.1000
Learner Weight (alpha_1):  2.1972

Updated & Normalized Sample Weights (w_2):
[0.05555556 0.05555556 0.05555556 0.05555556 0.05555556 0.05555556
 0.5        0.05555556 0.05555556 0.05555556]

